In [ ]:
# =========================
# LightGCL: Simple Yet Effective Graph Contrastive Learning for Recommendation
# Full Implementation in One Cell for Kaggle/Colab
# =========================

import os
import argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import pickle
import pandas as pd
from tqdm import tqdm
import time
import requests
import zipfile
import io

# -------------------------
# Step 1: Data Downloading
# -------------------------
# We clone the repo nicely to get the data, but run code defined here.
if not os.path.exists('LightGCL'):
    print("Cloning LightGCL repository for data...")
    os.system('git clone https://github.com/yangzeha/LightGCL.git')
else:
    print("LightGCL repository already exists.")

# -------------------------
# Step 2: Configuration (Args)
# -------------------------
class Args:
    def __init__(self):
        self.lr = 1e-3
        self.decay = 0.99
        self.batch = 256
        self.inter_batch = 4096
        self.note = None
        self.lambda1 = 1e-7
        self.epoch = 100
        self.d = 64
        self.q = 5
        self.gnn_layer = 2
        self.data = 'gowalla' # Default dataset
        self.dropout = 0.0
        self.temp = 0.1 # Based on main.py snippet (0.2 in parse_args default, but 0.1 in main runner example)
        self.lambda2 = 1e-7
        self.cuda = '0'

args = Args()

# Set device
if torch.cuda.is_available():
    device = 'cuda:' + args.cuda
else:
    device = 'cpu'

# -------------------------
# Step 3: Utils
# -------------------------
def metrics(uids, predictions, topk, test_labels):
    user_num = 0
    all_recall = 0
    all_ndcg = 0
    for i in range(len(uids)):
        uid = uids[i]
        prediction = list(predictions[i][:topk])
        label = test_labels[uid]
        if len(label)>0:
            hit = 0
            idcg = np.sum([np.reciprocal(np.log2(loc + 2)) for loc in range(min(topk, len(label)))])
            dcg = 0
            for item in label:
                if item in prediction:
                    hit+=1
                    loc = prediction.index(item)
                    dcg = dcg + np.reciprocal(np.log2(loc+2))
            all_recall = all_recall + hit/len(label)
            all_ndcg = all_ndcg + dcg/idcg
            user_num+=1
    return all_recall/user_num, all_ndcg/user_num

def scipy_sparse_mat_to_torch_sparse_tensor(sparse_mx):
    sparse_mx = sparse_mx.tocoo().astype(np.float32)
    indices = torch.from_numpy(
        np.vstack((sparse_mx.row, sparse_mx.col)).astype(np.int64))
    values = torch.from_numpy(sparse_mx.data)
    shape = torch.Size(sparse_mx.shape)
    return torch.sparse.FloatTensor(indices, values, shape)

def sparse_dropout(mat, dropout):
    if dropout == 0.0:
        return mat
    indices = mat.indices()
    values = nn.functional.dropout(mat.values(), p=dropout)
    size = mat.size()
    return torch.sparse.FloatTensor(indices, values, size)

class TrnData(data.Dataset):
    def __init__(self, coomat):
        self.rows = coomat.row
        self.cols = coomat.col
        self.dokmat = coomat.todok()
        self.negs = np.zeros(len(self.rows)).astype(np.int32)

    def neg_sampling(self):
        for i in range(len(self.rows)):
            u = self.rows[i]
            while True:
                i_neg = np.random.randint(self.dokmat.shape[1])
                if (u, i_neg) not in self.dokmat:
                    break
            self.negs[i] = i_neg

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx], self.cols[idx], self.negs[idx]

# -------------------------
# Step 4: Model
# -------------------------
class LightGCL(nn.Module):
    def __init__(self, n_u, n_i, d, u_mul_s, v_mul_s, ut, vt, train_csr, adj_norm, l, temp, lambda_1, lambda_2, dropout, batch_user, device):
        super(LightGCL,self).__init__()
        self.E_u_0 = nn.Parameter(nn.init.xavier_uniform_(torch.empty(n_u,d)))
        self.E_i_0 = nn.Parameter(nn.init.xavier_uniform_(torch.empty(n_i,d)))

        self.train_csr = train_csr
        self.adj_norm = adj_norm
        self.l = l
        self.E_u_list = [None] * (l+1)
        self.E_i_list = [None] * (l+1)
        self.E_u_list[0] = self.E_u_0
        self.E_i_list[0] = self.E_i_0
        self.Z_u_list = [None] * (l+1)
        self.Z_i_list = [None] * (l+1)
        self.G_u_list = [None] * (l+1)
        self.G_i_list = [None] * (l+1)
        self.G_u_list[0] = self.E_u_0
        self.G_i_list[0] = self.E_i_0
        self.temp = temp
        self.lambda_1 = lambda_1
        self.lambda_2 = lambda_2
        self.dropout = dropout
        self.act = nn.LeakyReLU(0.5)
        self.batch_user = batch_user

        self.E_u = None
        self.E_i = None

        self.u_mul_s = u_mul_s
        self.v_mul_s = v_mul_s
        self.ut = ut
        self.vt = vt

        self.device = device

    def forward(self, uids, iids, pos, neg, test=False):
        if test==True:  # testing phase
            preds = self.E_u[uids] @ self.E_i.T
            mask = self.train_csr[uids.cpu().numpy()].toarray()
            mask = torch.Tensor(mask).to(torch.device(self.device))
            preds = preds * (1-mask) - 1e8 * mask
            predictions = preds.argsort(descending=True)
            return predictions
        else:  # training phase
            for layer in range(1,self.l+1):
                # GNN propagation
                self.Z_u_list[layer] = (torch.spmm(sparse_dropout(self.adj_norm,self.dropout), self.E_i_list[layer-1]))
                self.Z_i_list[layer] = (torch.spmm(sparse_dropout(self.adj_norm,self.dropout).transpose(0,1), self.E_u_list[layer-1]))

                # svd_adj propagation
                vt_ei = self.vt @ self.E_i_list[layer-1]
                self.G_u_list[layer] = (self.u_mul_s @ vt_ei)
                ut_eu = self.ut @ self.E_u_list[layer-1]
                self.G_i_list[layer] = (self.v_mul_s @ ut_eu)

                # aggregate
                self.E_u_list[layer] = self.Z_u_list[layer]
                self.E_i_list[layer] = self.Z_i_list[layer]

            self.G_u = sum(self.G_u_list)
            self.G_i = sum(self.G_i_list)

            # aggregate across layers
            self.E_u = sum(self.E_u_list)
            self.E_i = sum(self.E_i_list)

            # cl loss
            G_u_norm = self.G_u
            E_u_norm = self.E_u
            G_i_norm = self.G_i
            E_i_norm = self.E_i
            neg_score = torch.log(torch.exp(G_u_norm[uids] @ E_u_norm.T / self.temp).sum(1) + 1e-8).mean()
            neg_score += torch.log(torch.exp(G_i_norm[iids] @ E_i_norm.T / self.temp).sum(1) + 1e-8).mean()
            pos_score = (torch.clamp((G_u_norm[uids] * E_u_norm[uids]).sum(1) / self.temp,-5.0,5.0)).mean() + (torch.clamp((G_i_norm[iids] * E_i_norm[iids]).sum(1) / self.temp,-5.0,5.0)).mean()
            loss_s = -pos_score + neg_score

            # bpr loss
            u_emb = self.E_u[uids]
            pos_emb = self.E_i[pos]
            neg_emb = self.E_i[neg]
            pos_scores = (u_emb * pos_emb).sum(-1)
            neg_scores = (u_emb * neg_emb).sum(-1)
            loss_r = -(pos_scores - neg_scores).sigmoid().log().mean()

            # reg loss
            loss_reg = 0
            for param in self.parameters():
                loss_reg += param.norm(2).square()
            loss_reg *= self.lambda_2

            # total loss
            loss = loss_r + self.lambda_1 * loss_s + loss_reg
            #print('loss',loss.item(),'loss_r',loss_r.item(),'loss_s',loss_s.item())
            return loss, loss_r, self.lambda_1 * loss_s

# -------------------------
# Step 5: Main Training Loop
# -------------------------

# hyperparameters
d = args.d
l = args.gnn_layer
temp = args.temp
batch_user = args.batch
epoch_no = args.epoch
max_samp = 40
lambda_1 = args.lambda1
lambda_2 = args.lambda2
dropout = args.dropout
lr = args.lr
decay = args.decay
svd_q = args.q

# Load Data
# Note: Data path needs to be correct relative to where we are.
# If we cloned into LightGCL/, the data is at LightGCL/data/
# But if we run this elsewhere, we need to handle it.
data_path = 'LightGCL/data/' + args.data + '/'
if not os.path.exists(data_path):
    # Try looking in local data/ if not in subfolder
    data_path = 'data/' + args.data + '/'

print(f"Loading data from {data_path}")

try:
    with open(data_path+'trnMat.pkl','rb') as f:
        train = pickle.load(f)
    train_csr = (train!=0).astype(np.float32)
    with open(data_path+'tstMat.pkl','rb') as f:
        test = pickle.load(f)
    print('Data loaded.')
except FileNotFoundError:
    print("Error: Data files not found. Ensure the repository is cloned and data is available.")
    # Exit or stop execution
    # raise Exception("Data not found")
    # For safe execution in One Cell context, we might want to stop here
    train = None

if train is not None:
    print('user_num:',train.shape[0],'item_num:',train.shape[1],'lambda_1:',lambda_1,'lambda_2:',lambda_2,'temp:',temp,'q:',svd_q)

    epoch_user = min(train.shape[0], 30000)

    # normalizing the adj matrix
    rowD = np.array(train.sum(1)).squeeze()
    colD = np.array(train.sum(0)).squeeze()
    for i in range(len(train.data)):
        train.data[i] = train.data[i] / pow(rowD[train.row[i]]*colD[train.col[i]], 0.5)

    # construct data loader
    train = train.tocoo()
    train_data = TrnData(train)
    train_loader = data.DataLoader(train_data, batch_size=args.inter_batch, shuffle=True, num_workers=0)

    adj_norm = scipy_sparse_mat_to_torch_sparse_tensor(train)
    adj_norm = adj_norm.coalesce().to(torch.device(device))
    print('Adj matrix normalized.')

    # perform svd reconstruction
    adj = scipy_sparse_mat_to_torch_sparse_tensor(train).coalesce().to(torch.device(device))
    print('Performing SVD...')
    svd_u,s,svd_v = torch.svd_lowrank(adj, q=svd_q)
    u_mul_s = svd_u @ (torch.diag(s))
    v_mul_s = svd_v @ (torch.diag(s))
    del s
    print('SVD done.')

    # process test set
    test_labels = [[] for i in range(test.shape[0])]
    for i in range(len(test.data)):
        row = test.row[i]
        col = test.col[i]
        test_labels[row].append(col)
    print('Test data processed.')

    loss_list = []
    loss_r_list = []
    loss_s_list = []
    recall_20_x = []
    recall_20_y = []
    ndcg_20_y = []
    recall_40_y = []
    ndcg_40_y = []

    model = LightGCL(adj_norm.shape[0], adj_norm.shape[1], d, u_mul_s, v_mul_s, svd_u.T, svd_v.T, train_csr, adj_norm, l, temp, lambda_1, lambda_2, dropout, batch_user, device)
    model.to(torch.device(device))
    optimizer = torch.optim.Adam(model.parameters(),weight_decay=0,lr=lr)

    print("Starting Training...")
    for epoch in range(epoch_no):
        epoch_loss = 0
        epoch_loss_r = 0
        epoch_loss_s = 0
        train_loader.dataset.neg_sampling()
        
        # Use simple progress print or custom loop to avoid tqdm issues in some envs, but tqdm is standard
        for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}")):
            uids, pos, neg = batch
            uids = uids.long().to(torch.device(device))
            pos = pos.long().to(torch.device(device))
            neg = neg.long().to(torch.device(device))
            iids = torch.concat([pos, neg], dim=0)

            # feed
            optimizer.zero_grad()
            loss, loss_r, loss_s= model(uids, iids, pos, neg)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.cpu().item()
            epoch_loss_r += loss_r.cpu().item()
            epoch_loss_s += loss_s.cpu().item()

            torch.cuda.empty_cache()

        batch_no = len(train_loader)
        epoch_loss = epoch_loss/batch_no
        epoch_loss_r = epoch_loss_r/batch_no
        epoch_loss_s = epoch_loss_s/batch_no
        loss_list.append(epoch_loss)
        loss_r_list.append(epoch_loss_r)
        loss_s_list.append(epoch_loss_s)
        print('Epoch:',epoch,'Loss:',epoch_loss,'Loss_r:',epoch_loss_r,'Loss_s:',epoch_loss_s)

        if epoch % 3 == 0:  # test every 3 epochs
            test_uids = np.array([i for i in range(adj_norm.shape[0])])
            batch_no = int(np.ceil(len(test_uids)/batch_user))

            all_recall_20 = 0
            all_ndcg_20 = 0
            all_recall_40 = 0
            all_ndcg_40 = 0
            for batch in range(batch_no):
                start = batch*batch_user
                end = min((batch+1)*batch_user,len(test_uids))

                test_uids_input = torch.LongTensor(test_uids[start:end]).to(torch.device(device))
                predictions = model(test_uids_input,None,None,None,test=True)
                predictions = np.array(predictions.cpu())

                #top@20
                recall_20, ndcg_20 = metrics(test_uids[start:end],predictions,20,test_labels)
                #top@40
                recall_40, ndcg_40 = metrics(test_uids[start:end],predictions,40,test_labels)

                all_recall_20+=recall_20
                all_ndcg_20+=ndcg_20
                all_recall_40+=recall_40
                all_ndcg_40+=ndcg_40
            
            print('-------------------------------------------')
            print('Test of epoch',epoch,':','Recall@20:',all_recall_20/batch_no,'Ndcg@20:',all_ndcg_20/batch_no,'Recall@40:',all_recall_40/batch_no,'Ndcg@40:',all_ndcg_40/batch_no)
            recall_20_x.append(epoch)
            recall_20_y.append(all_recall_20/batch_no)
            ndcg_20_y.append(all_ndcg_20/batch_no)
            recall_40_y.append(all_recall_40/batch_no)
            ndcg_40_y.append(all_ndcg_40/batch_no)

    print("Training Finished.")

# LightGCL Execution on Kaggle/Colab

This notebook is designed to run the LightGCL model directly from the GitHub repository.

Detailed instructions:
1.  Clone the repository.
2.  Navigate to the directory.
3.  Run the main script.

In [ ]:
# Install necessary libraries (if using Kaggle/Colab, these might be pre-installed)
# !pip install torch numpy pandas

# Clone the LightGCL repository
!git clone https://github.com/yangzeha/LightGCL.git

# Change directory to the cloned repository
%cd LightGCL

# Run the main script
# You can modify the arguments here (e.g., --data gowalla, --epoch 100)
!python main.py --data gowalla --lambda1 1e-7 --lambda2 1e-7 --temp 0.1 --q 5

In [ ]:
# Clone the repository if running in Google Colab or similar environment
!git clone https://github.com/yangzeha/LightGCL.git
%cd LightGCL

# LightGCL: Simple Yet Effective Graph Contrastive Learning for Recommendation

This notebook implements the LightGCL model for recommendation. It is based on the `main.py` script but adapted for interactive execution.

GitHub Repository: [https://github.com/yangzeha/LightGCL](https://github.com/yangzeha/LightGCL)

In [ ]:
import numpy as np
import torch
import pickle
from model import LightGCL
from utils import metrics, scipy_sparse_mat_to_torch_sparse_tensor, TrnData
import pandas as pd
from tqdm import tqdm
import time
import torch.utils.data as data
import os

# ========================== Configuration ==========================
class Args:
    def __init__(self):
        self.lr = 1e-3
        self.decay = 0.99
        self.batch = 256
        self.inter_batch = 4096
        self.note = None
        self.lambda1 = 0.2
        self.lambda2 = 1e-7
        self.epoch = 100
        self.d = 64
        self.q = 5
        self.gnn_layer = 2
        self.data = 'yelp'
        self.dropout = 0.0
        self.temp = 0.2
        self.cuda = '0'

args = Args()

# Check device
if torch.cuda.is_available():
    device = 'cuda:' + args.cuda
else:
    device = 'cpu'
print(f"Using device: {device}")

# hyperparameters
d = args.d
l = args.gnn_layer
temp = args.temp
batch_user = args.batch
epoch_no = args.epoch
max_samp = 40
lambda_1 = args.lambda1
lambda_2 = args.lambda2
dropout = args.dropout
lr = args.lr
decay = args.decay
svd_q = args.q

# ========================== Data Loading ==========================
path = 'data/' + args.data + '/'
print(f"Loading data from {path}...")
f = open(path+'trnMat.pkl','rb')
train = pickle.load(f)
train_csr = (train!=0).astype(np.float32)
f = open(path+'tstMat.pkl','rb')
test = pickle.load(f)
print('Data loaded.')
print('user_num:',train.shape[0],'item_num:',train.shape[1],'lambda_1:',lambda_1,'lambda_2:',lambda_2,'temp:',temp,'q:',svd_q)

epoch_user = min(train.shape[0], 30000)

# normalizing the adj matrix
rowD = np.array(train.sum(1)).squeeze()
colD = np.array(train.sum(0)).squeeze()
for i in range(len(train.data)):
    train.data[i] = train.data[i] / pow(rowD[train.row[i]]*colD[train.col[i]], 0.5)

# construct data loader
train = train.tocoo()
train_data = TrnData(train)
train_loader = data.DataLoader(train_data, batch_size=args.inter_batch, shuffle=True, num_workers=0)

adj_norm = scipy_sparse_mat_to_torch_sparse_tensor(train)
adj_norm = adj_norm.coalesce().to(torch.device(device))
print('Adj matrix normalized.')

# perform svd reconstruction
adj = scipy_sparse_mat_to_torch_sparse_tensor(train).coalesce().to(torch.device(device))
print('Performing SVD...')
svd_u,s,svd_v = torch.svd_lowrank(adj, q=svd_q)
u_mul_s = svd_u @ (torch.diag(s))
v_mul_s = svd_v @ (torch.diag(s))
del s
print('SVD done.')

# process test set
test_labels = [[] for i in range(test.shape[0])]
for i in range(len(test.data)):
    row = test.row[i]
    col = test.col[i]
    test_labels[row].append(col)
print('Test data processed.')

# ========================== Model & Training ==========================
loss_list = []
loss_r_list = []
loss_s_list = []
recall_20_x = []
recall_20_y = []
ndcg_20_y = []
recall_40_y = []
ndcg_40_y = []

model = LightGCL(adj_norm.shape[0], adj_norm.shape[1], d, u_mul_s, v_mul_s, svd_u.T, svd_v.T, train_csr, adj_norm, l, temp, lambda_1, lambda_2, dropout, batch_user, device)
#model.load_state_dict(torch.load('saved_model.pt'))
model.to(torch.device(device))
optimizer = torch.optim.Adam(model.parameters(),weight_decay=0,lr=lr)
#optimizer.load_state_dict(torch.load('saved_optim.pt'))

current_lr = lr

# Create directories if they don't exist
if not os.path.exists('log'):
    os.makedirs('log')
if not os.path.exists('saved_model'):
    os.makedirs('saved_model')

for epoch in range(epoch_no):
    if (epoch+1)%50 == 0:
        torch.save(model.state_dict(),'saved_model/saved_model_epoch_'+str(epoch)+'.pt')
        torch.save(optimizer.state_dict(),'saved_model/saved_optim_epoch_'+str(epoch)+'.pt')

    epoch_loss = 0
    epoch_loss_r = 0
    epoch_loss_s = 0
    train_loader.dataset.neg_sampling()
    
    # Wrap with tqdm for progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epoch_no}")
    for i, batch in enumerate(pbar):
        uids, pos, neg = batch
        uids = uids.long().to(torch.device(device))
        pos = pos.long().to(torch.device(device))
        neg = neg.long().to(torch.device(device))
        iids = torch.concat([pos, neg], dim=0)

        # feed
        optimizer.zero_grad()
        loss, loss_r, loss_s= model(uids, iids, pos, neg)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.cpu().item()
        epoch_loss_r += loss_r.cpu().item()
        epoch_loss_s += loss_s.cpu().item()

        if device.startswith('cuda'):
            torch.cuda.empty_cache()
        pbar.set_postfix({'Loss': loss.cpu().item()})

    batch_no = len(train_loader)
    epoch_loss = epoch_loss/batch_no
    epoch_loss_r = epoch_loss_r/batch_no
    epoch_loss_s = epoch_loss_s/batch_no
    loss_list.append(epoch_loss)
    loss_r_list.append(epoch_loss_r)
    loss_s_list.append(epoch_loss_s)
    print('Epoch:',epoch,'Loss:',epoch_loss,'Loss_r:',epoch_loss_r,'Loss_s:',epoch_loss_s)

    if epoch % 3 == 0:  # test every 3 epochs
        test_uids = np.array([i for i in range(adj_norm.shape[0])])
        batch_no_test = int(np.ceil(len(test_uids)/batch_user))

        all_recall_20 = 0
        all_ndcg_20 = 0
        all_recall_40 = 0
        all_ndcg_40 = 0
        for batch in tqdm(range(batch_no_test), desc="Testing"):
            start = batch*batch_user
            end = min((batch+1)*batch_user,len(test_uids))

            test_uids_input = torch.LongTensor(test_uids[start:end]).to(torch.device(device))
            predictions = model(test_uids_input,None,None,None,test=True)
            predictions = np.array(predictions.cpu())

            #top@20
            recall_20, ndcg_20 = metrics(test_uids[start:end],predictions,20,test_labels)
            #top@40
            recall_40, ndcg_40 = metrics(test_uids[start:end],predictions,40,test_labels)

            all_recall_20+=recall_20
            all_ndcg_20+=ndcg_20
            all_recall_40+=recall_40
            all_ndcg_40+=ndcg_40
            
        print('-------------------------------------------')
        print('Test of epoch',epoch,':','Recall@20:',all_recall_20/batch_no_test,'Ndcg@20:',all_ndcg_20/batch_no_test,'Recall@40:',all_recall_40/batch_no_test,'Ndcg@40:',all_ndcg_40/batch_no_test)
        recall_20_x.append(epoch)
        recall_20_y.append(all_recall_20/batch_no_test)
        ndcg_20_y.append(all_ndcg_20/batch_no_test)
        recall_40_y.append(all_recall_40/batch_no_test)
        ndcg_40_y.append(all_ndcg_40/batch_no_test)

# ========================== Final Evaluation ==========================
test_uids = np.array([i for i in range(adj_norm.shape[0])])
batch_no_test = int(np.ceil(len(test_uids)/batch_user))

all_recall_20 = 0
all_ndcg_20 = 0
all_recall_40 = 0
all_ndcg_40 = 0
for batch in range(batch_no_test):
    start = batch*batch_user
    end = min((batch+1)*batch_user,len(test_uids))

    test_uids_input = torch.LongTensor(test_uids[start:end]).to(torch.device(device))
    predictions = model(test_uids_input,None,None,None,test=True)
    predictions = np.array(predictions.cpu())

    #top@20
    recall_20, ndcg_20 = metrics(test_uids[start:end],predictions,20,test_labels)
    #top@40
    recall_40, ndcg_40 = metrics(test_uids[start:end],predictions,40,test_labels)

    all_recall_20+=recall_20
    all_ndcg_20+=ndcg_20
    all_recall_40+=recall_40
    all_ndcg_40+=ndcg_40

print('-------------------------------------------')
print('Final test:','Recall@20:',all_recall_20/batch_no_test,'Ndcg@20:',all_ndcg_20/batch_no_test,'Recall@40:',all_recall_40/batch_no_test,'Ndcg@40:',all_ndcg_40/batch_no_test)

recall_20_x.append('Final')
recall_20_y.append(all_recall_20/batch_no_test)
ndcg_20_y.append(all_ndcg_20/batch_no_test)
recall_40_y.append(all_recall_40/batch_no_test)
ndcg_40_y.append(all_ndcg_40/batch_no_test)

metric = pd.DataFrame({
    'epoch':recall_20_x,
    'recall@20':recall_20_y,
    'ndcg@20':ndcg_20_y,
    'recall@40':recall_40_y,
    'ndcg@40':ndcg_40_y
})
current_t = time.gmtime()
metric.to_csv('log/result_'+args.data+'_'+time.strftime('%Y-%m-%d-%H',current_t)+'.csv')

torch.save(model.state_dict(),'saved_model/saved_model_'+args.data+'_'+time.strftime('%Y-%m-%d-%H',current_t)+'.pt')
torch.save(optimizer.state_dict(),'saved_model/saved_optim_'+args.data+'_'+time.strftime('%Y-%m-%d-%H',current_t)+'.pt')

## Configuration

Set the hyperparameters here. You can change `data` to 'gowalla', 'yelp', etc.

## Data Loading

## Data Preprocessing & SVD

## Model Initialization

## Training Loop

## Final Evaluation & Saving Results